In [3]:
import numpy as np

# Tools for Finding Quantum Advantage

In this notebook we will demo some custom tools for determining whether a nonlocal game has quantum advantage.
This will be useful as you model different HFT scenarios as nonlocal games.

## Defining a nonlocal game
A nonlocal game consists of two arrays, a utility array `V[x, y, a, b]` representing the payoff when Alice and Bob take joint action (a,b) in response to observations (x,y) and an array `pi[x, y]` representing the probability distribution over the possible observations (x,y).
For example, here is the CHSH game:

In [4]:
pi = np.ones((2, 2))/4

V = np.array([
    # (a, b)
    # (0, 0), (0, 1), (1, 0), (1, 1)
                                            # (x, y)
    [      1,      0,      0,      1],      # (0, 0)
    [      1,      0,      0,      1],      # (0, 1)
    [      1,      0,      0,      1],      # (1, 0)
    [      0,      1,      1,      0]       # (1, 1)

]).reshape((2, 2, 2, 2))

## Find classical best
The _classical value_ of a game is the best score you can get without making use of entanglement.
You can show that the classical value can always be obtained by some deterministic strategy.
So one method to find the classical value is to simply enumerate over every deterministic strategy that Alice and Bob can implement.
We have a function that does this for you.
Let's try it on the CHSH game.
It should recover the known classical best of 0.75.

In [5]:
from classical_value import classical_value as find_classical_value

classical_value = find_classical_value(pi, V)
classical_value

0.75

## Find quantum best
In general it is a hard optimization problem to find the best quantum strategy for a nonlocal game.
Instead, one can use various methods to find **lower bounds**.
We have a custom implementation of what's called the "see-saw" method, which works pretty well for small games.
Let's try it on the CHSH game

In [6]:
from lower_bounds import seesaw

quantum_lower_bound = seesaw(pi, V, dim=2)
quantum_lower_bound

0.8535533899306176

## Upper bounds on the quantum value
While the see-saw method gives lower bounds on the quantum value of a game, there are various methods for getting upper bounds.
In particular, all strategies implementable by Alice and Bob must satisfy what are called the no-signalling condition.
This constraint on their strategy gives an easy to calculate upper bound.
This can often be used to very quickly rule out quantum advantage if this upper bound corresponds with the classical value!
We implemented this as a function you can use.
Let's try it on the CHSH game:

In [7]:
from upper_bounds import nonsignaling_value

quantum_upper_bound = nonsignaling_value(pi, V)
quantum_upper_bound

1.0

## Quantifying quantum advantage
One standard way to quantify how much quantum advantage there is for a nonlocal game is to first offset the classical and quantum best scores by the score you'd get with a completely random strategy, then to relativize the gap between quantum and classical by the newly offset classical value:

In [8]:
random_value = (pi[:, :, None, None] * V).mean(axis=(2, 3)).sum()  # Alice and Bob answer uniformly at random

quantum_advantage = (quantum_lower_bound - classical_value)/(classical_value - random_value)
quantum_advantage

np.float64(0.4142135597224703)

## Example

In [16]:
# Does this game have a quantum advantage? It is a nonlocal game where x 
# and y can each take three different values and where a and b can each 
# take two different values.

V = np.array([
    [ 1, -1, -1,  1], # (x, y) = (0, 0)
    [-1,  1, -1, -1], # (x, y) = (0, 1)
    [-1, -1,  1, -1], # (x, y) = (0, 2)
    [-1, -1, -1,  1], # (x, y) = (1, 0)
    [ 1, -1, -1, -1], # (x, y) = (1, 1)
    [-1,  1, -1, -1], # (x, y) = (1, 2)
    [ 1,  1,  1, -1], # (x, y) = (2, 0)
    [-1, -1,  1,  1], # (x, y) = (2, 1)
    [ 1,  1, -1, -1]  # (x, y) = (2, 2)
]).reshape((3, 3, 2, 2))

pi = np.ones((3, 3))/9


# Let's compute the classical value of this game:
classical_value = find_classical_value(pi, V)
print(f"Classical best value: {classical_value}")

# Now let's get an upper bound on the quantum value by calculating the nonsignaling value:
quantum_upper_bound = nonsignaling_value(pi, V)
print(f"Upper bound on quantum best: {quantum_upper_bound}")

# Now let's get a lower bound on the quantum value by running the seesaw algorithm:
quantum_lower_bound = seesaw(pi, V, dim=2)
print(f"Lower bound on quantum best: {quantum_lower_bound}")

# Let's calculate a standardized quantum advantage metric:
random_value = (pi[:, :, None, None] * V).mean(axis=(2, 3)).sum()  # Alice and Bob answer uniformly at random
quantum_advantage = (quantum_lower_bound - classical_value)/(classical_value - random_value)
print(f"Quantum advantage: {quantum_advantage*100:.2f}%")

Classical best value: 0.1111111111111111
Upper bound on quantum best: 0.22222222222222224
Lower bound on quantum best: 0.13681742764798688
Quantum advantage: 7.71%
